# Attributions for RAGs

RAGs (Retrieval Augmented Generation) adds long context to produced based generations.

The long context can be challenging to explain due to its sheer number of tokens. This use-case illustrate how interpreto's `Granularity` can be useful for generation tasks.

## 0. Setup

### 0.1 Setting up the environment

We use `uv` because it is faster, but you can use `pip`.

In [ ]:
# download uv
# pip install uv

# create the env
# uv venv

# activate the env
# source .venv/bin/activate  # linux
# .venv/Scripts/activate  # windows

# install required packages
# !uv pip install datasets interpreto

In [ ]:
# verify our installation
from time import perf_counter

import torch
from IPython.display import Markdown, display
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed

from interpreto import Granularity, Lime, Occlusion, plot_attributions

In [ ]:
if not torch.cuda.is_available():
    print("Warning: CUDA is not available. This may become quite long, we recommend tuning down the parameters.")
    device = "cpu"
else:
    device = "cuda"

### 0.2 Load the model

Let's load a small generation model from HuggingFace.

In [ ]:
model_id = 'Qwen/Qwen3.5-0.8B'
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, dtype='auto', device_map=device).eval()

### 0.3 Simulate retrieved context

Each hand-written chunk is one sentence, so sentence granularity lets us compare whole chunks. They paraphrase [Interpreto Language Services](https://getinterpreto.com/), the [Latin root](https://en.wiktionary.org/wiki/interpretor), the [Italian verb](https://en.wiktionary.org/wiki/interpretare), [our library](https://github.com/FOR-sight-ai/interpreto), and [Interpreto's psychometric tests](https://www.interpreto.ca/en/).

In [ ]:
chunks = [
    '1. Interpreto Language Services connects clients with professional interpreters for healthcare, legal, and business appointments, helping people who speak different languages communicate during situations such as a medical consultation, a discussion of legal paperwork, or a meeting with a commercial partner, where the interpreter conveys the meaning of what participants say, preserves relevant details and tone, and supports mutual understanding so that language differences do not prevent people from asking questions, expressing concerns, or following the conversation.',
    '2. The name interpreto evokes the Latin verb interpretari, meaning to interpret or explain, a root associated with making something understandable by clarifying its meaning, expressing an idea in other words, or helping an audience understand a message, and this linguistic connection makes the name suggest explanation and understanding across several otherwise unrelated activities, from working with languages to discussing texts or describing complex systems.',
    '3. In Italian, interpreto means I interpret and is the first-person singular present of interpretare, a verb that can describe explaining the meaning of a statement, understanding an event from a particular perspective, translating spoken language, or performing a role in a play or film, so the intended sense depends on the surrounding words and situation rather than identifying one particular company, software package, profession, or technical method on its own.',
    '4. Interpreto is a Python toolkit for interpreting transformer models through attribution and concept-based explanations, with tutorial notebooks and visualizations that guide users from loading a model and preparing text inputs to examining which parts of an input influence a classification or generated answer, while its concept methods help explore patterns in internal activations and its configurable granularity lets explanations focus on tokens, words, or sentences, making the library useful for practical experiments in model interpretability and for teaching how different explanation methods reveal aspects of model behavior through reusable components and concrete examples.',
    '5. Interpreto also names an online psychometric testing service for recruitment and professional development, placing the name in a human resources setting where assessments can help structure discussions about candidates, employees, and workplace development goals, with results serving as material for considering individual profiles alongside interviews and other information, rather than referring to the interpretation of a transformer model or the translation of a spoken conversation.',
]

In [ ]:
question = 'I got a tutorial notebook, in this context, do you have any idea of what interpreto is?'
prompt = (
    'Answer the question using the retrieved information chunks.\n\nInformation chunks:\n'
    + '\n'.join(chunks) + f'\n\nQuestion: {question}\n\nAnswer:'
)

### 0.4 Generate an answer

We use chat template and thinking to generate a realistic answer to the question.

In [ ]:
messages = [
    {"role": "user", "content": prompt}
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    enable_thinking=True,
    return_dict=True,
    return_tensors="pt",
).to(model.device)

generated = model.generate(
    **inputs,
    max_new_tokens=2048,
    do_sample=True,
    temperature=1.0,
    top_p=0.95,
    top_k=20,
    repetition_penalty=1.2,
)

generated = generated[0, inputs.input_ids.shape[1]:]

print(tokenizer.decode(generated, skip_special_tokens=False))

## 1. Explainability

The above input contains both a long context and long reasoning trace. The reasoning trace is part of the generated output. Such sample is tricky to explain due to the number of tokens and thus the combinatorial number of perturbations required to have a converging explanation.

Fortunately, interpreto's `Granularity.SENTENCE` groups tokens by sentences in both inputs and outputs. Which considerably reduces the number of elements to perturb. We could see it as explaining patches in images rather than pixels.

To show the time or quality gain, we explore two methods, `Lime` and `Occlusion`.

In [ ]:
batch_size = 4
explanations = {}

### 1.1 Occlusion

Occlusion perturbs one element at a time. So the number of perturbations is equal to the number of elements.

We therefore compare the perturbations by `Granularity.WORD` (default) and `Granularity.SENTENCE`.

In [ ]:
start = perf_counter()

# Compute default Occlusion
explainer = Occlusion(model, tokenizer, batch_size=4)
explanations['Occlusion: default (WORD)'] = explainer(prompt, targets=generated)[0]

between = perf_counter()
print(f'Occlusion default: {between - start:.2f} s')

# Compute Occlusion with sentence granularity
explainer = Occlusion(
    model, tokenizer, batch_size=4, granularity=Granularity.SENTENCE
)
explanations['Occlusion: SENTENCE'] = explainer(prompt, targets=generated)[0]

end = perf_counter()
print(f'Occlusion SENTENCE: {end - between:.2f} s')

In [ ]:
for label, explanation in explanations.items():
    display(Markdown(f'#### {label}'))
    plot_attributions(explanation)

### 1.2 Lime

Lime takes a `n_perturbations` which is $100$ by default. However, compared to the number of words in the input, this will not allow the explanation to converge. We will see this qualitatively. (removed because of file size).

Therefore, we compare it with the same granularity (`WORD`) and $4000$ perturbations.

Finally, we try the `SENTENCE` granularity with additional perturbations, $500$.

In [ ]:
set_seed(42)

start = perf_counter()

# # Compute default Lime
explainer = Lime(model, tokenizer, batch_size=batch_size)
explanations['Lime: WORD, 100 (default)'] = explainer(prompt, targets=generated)[0]

after_default = perf_counter()
print(f'Lime default: {after_default - start:.2f} s')

# Compute Lime with 4000 perturbations
explainer = Lime(
    model, tokenizer, batch_size=batch_size, n_perturbations=4000
)
explanations['Lime: WORD, 4000'] = explainer(prompt, targets=generated)[0]

after_4000 = perf_counter()
print(f'Lime 4000 perturbations: {after_4000 - after_default:.2f} s')

# Compute Lime with sentence granularity
explainer = Lime(
    model, tokenizer, batch_size=batch_size, granularity=Granularity.SENTENCE, n_perturbations=500
)
explanations['Lime: SENTENCE, 500'] = explainer(prompt, targets=generated)[0]

end = perf_counter()
print(f'Lime SENTENCE, 500 perturbations: {end - after_4000:.2f} s')

In [ ]:
label = 'Lime: SENTENCE, 500'
display(Markdown(f'#### {label}'))
plot_attributions(explanations[label])